> **LangChain 1.x / 2026** — *2026** — drug-discovery evidence is auditable and human-gated; optional paid LLM only where noted. See `UPDATE_2026.md`.

# Chapter 8 — Generative Design Validation Gates (v2026) (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2008.%20LangChain%20for%20Drug%20Discovery/LC4LSH_Chapter_8_Generative_Design_Validation_Gates.ipynb)

**Learning objectives**
- Validate generated structures BEFORE ranking
- Check validity, uniqueness, novelty, properties, structural alerts
- Attach uncertainty and require evidence support
- Block any candidate that fails a gate from the shortlist

> Runtime: ~5 min (local, RDKit)  
> Cost: free  
> Data: small built-in generated-SMILES list

## Environment setup

### Secrets (optional LLM only)

In [ ]:
import os


def get_secret(name, default=None):
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass
    return os.environ.get(name, default)


# These notebooks are LOCAL-first (RDKit/pandas/sklearn); a paid LLM is OPTIONAL.
OPENAI_API_KEY = get_secret("LC4LS_OPENAI_API_KEY", "")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("OpenAI key set (optional):", bool(OPENAI_API_KEY))

### Install pinned dependencies

In [ ]:
%pip install -q rdkit "pandas>=2.0" "numpy>=1.26" "matplotlib>=3.8" "scikit-learn>=1.4" "langchain==1.0.0" "langchain-openai==1.0.0" "python-dotenv>=1.0" # Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)

In [ ]:
# Optional LangSmith tracing (only if a key is present)
import os
LANGSMITH_API_KEY = get_secret("LANGCHAIN_API_KEY", "") or ""
LANGSMITH_PROJECT = "lc4lsh-chapter8-validation-gates"
if LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_API_KEY", LANGSMITH_API_KEY)
    os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)
    print("LangSmith tracing ON ->", LANGSMITH_PROJECT)
else:
    print("LangSmith OFF (no key) - fine; these notebooks are local-first.")

## Generated molecules are hypotheses

A generative model can emit invalid, duplicate, trivially-similar, or reactive structures. **Every** candidate must pass a sequence of validation gates before it is even ranked — and a passing candidate is still a hypothesis, not a drug candidate.

## 1. A batch of generated SMILES + training set

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, QED
from rdkit import DataStructs
from rdkit.Chem import AllChem

TRAIN = ["CCO", "c1ccccc1", "CC(=O)Oc1ccccc1C(=O)O"]  # tiny "training set" for novelty
GENERATED = ["CCO", "c1ccccc1", "CC(C)O", "O=C(O)c1ccccc1O", "C[C]C", "c1ccncc1", "not_a_smiles"]
print("generated:", GENERATED)

## 2. Gate 1 — validity

In [ ]:
def gate_valid(smiles):
    m = Chem.MolFromSmiles(smiles)
    return m

parsed = {s: gate_valid(s) for s in GENERATED}
for s, m in parsed.items():
    print(f"{'VALID  ' if m else 'INVALID'}  {s}")

## 3. Gate 2 — uniqueness & novelty

In [ ]:
def canon(m):
    return Chem.MolToSmiles(m)

train_canon = {canon(Chem.MolFromSmiles(s)) for s in TRAIN}
seen = set()
rows = []
for s, m in parsed.items():
    if not m:
        rows.append((s, "invalid", False, False)); continue
    c = canon(m)
    unique = c not in seen
    novel = c not in train_canon
    seen.add(c)
    rows.append((s, "ok", unique, novel))
for s, st, uniq, nov in rows:
    print(f"{s:22s} {st:8s} unique={uniq} novel={nov}")

## 4. Gate 3 — property range + uncertainty note

In [ ]:
def props(m):
    return {"mw": round(Descriptors.MolWt(m), 1),
            "logp": round(Descriptors.MolLogP(m), 2),
            "qed": round(QED.qed(m), 2)}

def prop_ok(p):
    return (150 <= p["mw"] <= 500) and (-1 <= p["logp"] <= 5)

for s, m in parsed.items():
    if not m: continue
    p = props(m)
    print(f"{s:22s} {p}  in_range={prop_ok(p)}")
print("\nNOTE: QED/descriptors are NOT evidence of activity/safety; property range is a soft gate only.")

## 5. Gate 4 — structural alerts (PAINS-like, illustrative)

In [ ]:
# Illustrative alert: flag some reactive/undesirable substructures (use FilterCatalog in practice)
ALERT_SMARTS = {"rhodanine": "S=C1NC(=O)CS1", "michael_acceptor": "C=CC=O"}
alert_patts = {k: Chem.MolFromSmarts(v) for k, v in ALERT_SMARTS.items()}

def alerts(m):
    return [k for k, p in alert_patts.items() if m.HasSubstructMatch(p)]

for s, m in parsed.items():
    if not m: continue
    print(f"{s:22s} alerts={alerts(m) or 'none'}")

## 6. Combine gates into a pass/hold decision

In [ ]:
def evaluate(smiles_list):
    seen = set(); out = []
    for s in smiles_list:
        m = Chem.MolFromSmiles(s)
        if not m:
            out.append((s, "HOLD", "invalid")); continue
        c = canon(m); p = props(m); a = alerts(m)
        reasons = []
        if c in seen: reasons.append("duplicate")
        if c in train_canon: reasons.append("not novel")
        if not prop_ok(p): reasons.append("out of property range")
        if a: reasons.append("alert:" + "/".join(a))
        seen.add(c)
        out.append((s, "PASS" if not reasons else "HOLD", "; ".join(reasons) or "all gates"))
    return out

for s, verdict, why in evaluate(GENERATED):
    print(f"{verdict:5s} {s:22s} {why}")
print("\nPASS = eligible for ranking (still a hypothesis). HOLD = blocked pending review.")

## Limitations & safety notes

- Alert SMARTS here are illustrative; use RDKit FilterCatalog (PAINS/BRENK/NIH) in practice.
- Property range + QED are soft filters, NOT evidence of activity, selectivity, or safety.
- Novelty is defined only against the tiny training set shown.
- Passing gates makes a candidate eligible for ranking — it is still a hypothesis requiring downstream validation + human review. Local/free; no LLM.

In [ ]:
import gc
gc.collect()
for _v in ["records", "df", "model", "llm", "X", "graph"]:
    globals().pop(_v, None)
gc.collect()
print("Cleanup done.")

## Exercises

<details><summary>Why validate before ranking?</summary>Ranking invalid/duplicate/reactive structures wastes review effort and inflates apparent hit rates; gates remove them first.</details>

<details><summary>Why is QED not a drug-likeness proof?</summary>QED is a heuristic desirability score over descriptors; it does not establish activity, safety, or synthesizability.</details>

<details><summary>Why block rather than just down-weight alerts?</summary>Some liabilities (reactive groups) are hard stops; blocking forces an explicit human decision instead of silent acceptance.</details>

### Tasks
- **Task A** - Replace the illustrative alerts with RDKit FilterCatalog and report PAINS/BRENK hit counts.
- **Task B** - Add a Tanimoto-similarity novelty threshold vs the training set (e.g., max sim < 0.6).
- **Task C** - Add a synthesizability estimate (SA score) as an additional gate.
- **Task D** - Emit a per-candidate validation report (each gate + pass/fail + reason) as JSON.